# Prediksi Harga Laptop - Preprocessing & Training (Linear Regression)

## 1) Import Library

In [ ]:
import pandas as pdimport numpy as npimport reimport seaborn as snsimport matplotlib.pyplot as pltimport joblibimport json

## 2) Upload & Load Dataset

In [ ]:
from google.colab import filesfiles.upload()  # pilih file laptop_price.csv

In [ ]:
df = pd.read_csv('laptop_price.csv', encoding='latin-1')df.head()

## 3) Eksplorasi Data Awal (EDA)

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
df['Price_euros'].hist(bins=30, figsize=(10,5), color='teal')plt.title('Distribusi Harga Laptop (Euro)')plt.show()

## 4) Data Preprocessing / Feature EngineeringKolom mentah (`Ram`, `Weight`, `Memory`, `Cpu`, `Gpu`, `ScreenResolution`, `OpSys`) berbentuk teksdan tidak bisa langsung dipakai regresi linier, sehingga perlu diubah menjadi angkaatau kategori sederhana.

### 4.1 Ram & Weight -> angka

In [ ]:
df['Ram'] = df['Ram'].str.replace('GB', '', regex=False).astype(int)df['Weight'] = df['Weight'].str.replace('kg', '', regex=False).astype(float)df[['Ram', 'Weight']].head()

### 4.2 Memory -> ukuran penyimpanan (GB) & tipe penyimpanan

In [ ]:
def extract_storage(memory):    memory = memory.replace('Flash Storage', 'FlashStorage')    primary = memory.split('+')[0].strip()    m = re.search(r'([\d\.]+)(GB|TB)', primary)    size = float(m.group(1))    if m.group(2) == 'TB':        size *= 1024    if 'SSD' in primary:        storage_type = 'SSD'    elif 'HDD' in primary:        storage_type = 'HDD'    elif 'FlashStorage' in primary:        storage_type = 'Flash Storage'    elif 'Hybrid' in primary:        storage_type = 'Hybrid'    else:        storage_type = 'Other'    return pd.Series([size, storage_type])df[['Storage_GB', 'Storage_Type']] = df['Memory'].apply(extract_storage)df[['Memory', 'Storage_GB', 'Storage_Type']].head()

### 4.3 Cpu -> brand & kecepatan (GHz)

In [ ]:
def extract_cpu_speed(cpu):    m = re.search(r'([\d\.]+)GHz', cpu)    return float(m.group(1)) if m else np.nandf['Cpu_Brand'] = df['Cpu'].apply(lambda x: x.split()[0])df['Cpu_Speed_GHz'] = df['Cpu'].apply(extract_cpu_speed)df[['Cpu', 'Cpu_Brand', 'Cpu_Speed_GHz']].head()

### 4.4 Gpu -> brand

In [ ]:
df['Gpu_Brand'] = df['Gpu'].apply(lambda x: x.split()[0])df[['Gpu', 'Gpu_Brand']].head()

### 4.5 ScreenResolution -> PPI, Touchscreen, IPS Panel

In [ ]:
def extract_wh(res):    m = re.search(r'(\d+)x(\d+)', res)    return int(m.group(1)), int(m.group(2))wh = df['ScreenResolution'].apply(extract_wh)df['Res_Width'] = wh.apply(lambda x: x[0])df['Res_Height'] = wh.apply(lambda x: x[1])df['PPI'] = ((df['Res_Width']**2 + df['Res_Height']**2) ** 0.5) / df['Inches']df['Touchscreen'] = df['ScreenResolution'].str.contains('Touchscreen').astype(int)df['IPS_Panel'] = df['ScreenResolution'].str.contains('IPS').astype(int)df[['ScreenResolution', 'PPI', 'Touchscreen', 'IPS_Panel']].head()

### 4.6 OpSys -> sederhanakan kategori

In [ ]:
def simplify_os(os_name):    if 'Windows' in os_name:        return 'Windows'    elif os_name in ['macOS', 'Mac OS X']:        return 'macOS'    elif 'Linux' in os_name:        return 'Linux'    elif os_name == 'No OS':        return 'No OS'    else:        return 'Other'df['OS'] = df['OpSys'].apply(simplify_os)df['OS'].value_counts()

### 4.7 Cek hasil preprocessing & buang kolom mentah yang sudah tidak dipakai

In [ ]:
feature_df = df[[    'Company', 'TypeName', 'Inches', 'Ram', 'Weight', 'OS',    'Cpu_Brand', 'Cpu_Speed_GHz', 'Gpu_Brand',    'Storage_GB', 'Storage_Type', 'PPI', 'Touchscreen', 'IPS_Panel',    'Price_euros']].copy()feature_df.isnull().sum()

In [ ]:
feature_df = feature_df.dropna()feature_df.head()

### 4.8 Korelasi fitur numerik terhadap harga

In [ ]:
numeric_cols = ['Inches','Ram','Weight','Cpu_Speed_GHz','Storage_GB','PPI','Touchscreen','IPS_Panel','Price_euros']corr = feature_df[numeric_cols].corr()plt.figure(figsize=(9,7))sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')plt.show()

## 5) One-Hot Encoding untuk kolom kategorikal`Company`, `TypeName`, `OS`, `Cpu_Brand`, `Gpu_Brand`, `Storage_Type` diubah menjadi kolom0/1 dengan `pd.get_dummies`. Daftar kolom hasil encoding disimpan agar aplikasi Streamlitnanti bisa menyusun input dengan urutan kolom yang sama persis seperti saat training.

In [ ]:
categorical_cols = ['Company', 'TypeName', 'OS', 'Cpu_Brand', 'Gpu_Brand', 'Storage_Type']# simpan pilihan kategori asli untuk dipakai di dropdown Streamlitcategory_options = {col: sorted(feature_df[col].unique().tolist()) for col in categorical_cols}category_options

In [ ]:
encoded_df = pd.get_dummies(feature_df, columns=categorical_cols)encoded_df.head()

## 6) Split Data Training & Testing

In [ ]:
from sklearn.model_selection import train_test_splitX = encoded_df.drop('Price_euros', axis=1)y = encoded_df['Price_euros']feature_columns = list(X.columns)  # urutan kolom ini WAJIB sama saat prediksi nantiX_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)X_train.shape, X_test.shape

## 7) Training Model Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegressionfrom sklearn.metrics import mean_squared_error, r2_scorelr = LinearRegression()lr.fit(X_train, y_train)

## 8) Evaluasi Model

In [ ]:
y_pred = lr.predict(X_test)mse = mean_squared_error(y_test, y_pred)rmse = mse ** 0.5r2 = r2_score(y_test, y_pred)print('MSE  :', round(mse, 2))print('RMSE :', round(rmse, 2), 'euro')print('R2   :', round(r2, 4))

In [ ]:
plt.figure(figsize=(8,6))plt.scatter(y_test, y_pred, alpha=0.6)plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')plt.xlabel('Harga Aktual (euro)')plt.ylabel('Harga Prediksi (euro)')plt.title('Aktual vs Prediksi')plt.show()

## 9) Simpan Model & Metadata (untuk dipakai di Streamlit)

In [ ]:
joblib.dump(lr, 'laptop_price_model.pkl')metadata = {    'feature_columns': feature_columns,    'category_options': category_options,    'numeric_ranges': {        'Inches': [float(feature_df['Inches'].min()), float(feature_df['Inches'].max())],        'Ram': [int(feature_df['Ram'].min()), int(feature_df['Ram'].max())],        'Weight': [float(feature_df['Weight'].min()), float(feature_df['Weight'].max())],        'Cpu_Speed_GHz': [float(feature_df['Cpu_Speed_GHz'].min()), float(feature_df['Cpu_Speed_GHz'].max())],        'Storage_GB': [float(feature_df['Storage_GB'].min()), float(feature_df['Storage_GB'].max())],    }}with open('model_metadata.json', 'w') as f:    json.dump(metadata, f, indent=2)print('Tersimpan: laptop_price_model.pkl & model_metadata.json')

## 10) Download File Model & MetadataDownload kedua file berikut lalu taruh satu folder dengan `app.py` (Streamlit).

In [ ]:
files.download('laptop_price_model.pkl')files.download('model_metadata.json')